In [6]:
import pandas  as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

Random_state = 42

In [7]:
df=sns.load_dataset('titanic')
print("Shape:",df.shape)
df.head()

Shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [8]:
print(df.columns.tolist())
df=df[["survived","pclass","sex","age","sibsp","parch","fare","embarked"]].copy()

df.head()


['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']


,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [9]:
print(df.isnull().sum())

survived      0
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
dtype: int64


In [10]:
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
print(df.isnull().sum())

survived    0
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
dtype: int64


In [11]:
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['sex'] = df['sex'].map({"male": 0,"female":1})
df['embarked']= df['embarked'].map({"S":0,"C":1,"Q":2})
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,family_size
0,0,3,0,22.0,1,0,7.2500,0,2
1,1,1,1,38.0,1,0,71.2833,1,2
2,1,3,1,26.0,0,0,7.9250,0,1
3,1,1,1,35.0,1,0,53.1000,0,2
4,0,3,0,35.0,0,0,8.0500,0,1


In [12]:
X = df[["pclass","sex","age","fare","family_size","embarked"]]
y = df["survived"]
print("Features Used: ",X.columns.tolist())
print("Target: Survived 0 - did not survived and 1 - survived")

Features Used:  ['pclass', 'sex', 'age', 'fare', 'family_size', 'embarked']
Target: Survived 0 - did not survived and 1 - survived


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training Rows: ", X_train.shape[0])
print("Testing Rows: ", X_test.shape[0])

Training Rows:  712
Testing Rows:  179


In [14]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "K Nearest Neighbors": KNeighborsClassifier(n_neighbors=5)
}
result = {}
for name, model in models.items():
    model.fit(X_train,y_train)
    prediction = model.predict(X_test)
    accuracy = accuracy_score(y_test,prediction)
    result[name] = accuracy
    print(f"{name:22s} accuracy: {accuracy:.4f}")


Logistic Regression    accuracy: 0.8101
Decision Tree          accuracy: 0.7765
Random Forest          accuracy: 0.8324
K Nearest Neighbors    accuracy: 0.7207


In [15]:
best_name = max(result, key=result.get)
best_model = models[best_name]

print(f"Best model: {best_name} (Accuracy: {result[best_name]:.4f})")

Best model: Random Forest (Accuracy: 0.8324)


In [17]:
y_pred = best_model.predict(X_test)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred,target_names=["Did not survie","Survived"]))


Confusion Matrix:
[[92 13]
 [17 57]]

Classification Report:
                precision    recall  f1-score   support

Did not survie       0.84      0.88      0.86       105
      Survived       0.81      0.77      0.79        74

      accuracy                           0.83       179
     macro avg       0.83      0.82      0.83       179
  weighted avg       0.83      0.83      0.83       179



In [18]:
if hasattr(best_model,"feature_importances_"):
    importance = pd.Series(best_model.feature_importances_,index=X.columns)
    print(importance.sort_values(ascending=False))

else:
    print(f"{best_name} doesn't expose feature_importances_directly.")

fare           0.281870
sex            0.268759
age            0.254724
pclass         0.082604
family_size    0.080837
embarked       0.031206
dtype: float64


In [21]:
import pandas as pd

def predict_new_passenger(model):
    print("Enter Passenger Details to Predict Survival on the Titanic")

    pclass = int(input("Passenger Class (1=1st, 2=2nd, 3=3rd): "))
    sex_input = input("Sex (male/female): ").strip().lower()
    age = float(input("Age: "))
    fare = float(input("Fare Paid: "))
    sibsp = int(input("Number of Siblings/Spouses Aboard: "))
    parch = int(input("Number of Parents/Children Aboard: "))
    embarked_input = input("Port (C/Q/S): ").strip().upper()

    family_size = sibsp + parch + 1

    # Use these only if your training used the same mapping
    sex = 1 if sex_input == "female" else 0
    embarked_map = {"S": 0, "C": 1, "Q": 2}
    embarked = embarked_map.get(embarked_input, 0)

    passenger = pd.DataFrame([{
        "pclass": pclass,
        "sex": sex,
        "age": age,
        "fare": fare,
        "family_size": family_size,
        "embarked": embarked
    }])

    prediction = model.predict(passenger)[0]

    result = "Survived" if prediction == 1 else "Did Not Survive"
    print("\nPrediction:", result)

    if hasattr(model, "predict_proba"):
        probability = model.predict_proba(passenger)[0][1]
        print(f"Survival Probability: {probability:.1%}")

# Predict
predict_new_passenger(best_model)

Enter Passenger Details to Predict Survival on the Titanic

Prediction: Survived
Survival Probability: 100.0%
